# 05 — Rollout evaluation (dynamic_dt)

3-step autoregressive rollout on **2023 test** data.

| Knob | Description |
|------|-------------|
| `RUN_ALL_DT` | **`True`** → evaluate all dt in `DT_DAYS_LIST` |
| `DT_DAYS` | Single dt when `RUN_ALL_DT=False` |
| `EXPORT_KEY` | Must match training window (uv, iD=2) |
| `GPU_POOL` | **`(7,)`** — all rollouts on cuda:7 (sequential) |

**Current batch (matches 04):** `DT_DAYS_LIST = [3, 23, 25, 27, 30]` → dt_hours 72, 552, 600, 648, 720

GPU map: all dt → **cuda:7** (jobs run one at a time)

**Original 8 dt (cuda:0–7):** commented in cell 1.

Horizons: **+dt**, **+2dt**, **+3dt**. Sample plots → `Test&Rollout/.../samples/sample_{i}.png`

In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MW_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "dynamic_dt" else (
    NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Magic_window" else NOTEBOOK_DIR / "Magic_window"
)
DT_DIR = MW_DIR / "dynamic_dt"
ROOT = MW_DIR.parent
for _p in (ROOT, MW_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import dynamic_dt.multi_dt_runner as _mdr
import dynamic_dt.rollout_runner as _rr
importlib.reload(_mdr)
importlib.reload(_rr)
from dynamic_dt.device_plan import (
    GPU_POOL_1_7,
    MID_DT_DAYS_LIST,
    configure_device_plan,
    format_dt_device_plan,
)
from dynamic_dt.dt_config import dt_hours, forecast_horizons_days
from dynamic_dt.window_catalog import get_window_spec, primary_export_component
from dynamic_dt.window_paths import test_prefix
from dynamic_dt.run_registry import resolve_model_checkpoint, run_dir, run_name, weight_decay_run_tag, RUN_TAG
from dynamic_dt.rollout_runner import rollout_one_dt
from dynamic_dt.multi_dt_runner import launch_rollout_all, print_job_summary

LOG_DIR = DT_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ============ USER KNOBS ============
EXPORT_KEY = "magicwin_reference"   # or "smallfixwin"

# --- Original 8 dt on cuda:0–7 (comment out mid-range block below to restore) ---
# from dynamic_dt.device_plan import STANDARD_DT_DAYS, clear_device_plan
# DT_DAYS_LIST = list(STANDARD_DT_DAYS)
# GPU_POOL = tuple(range(8))
# clear_device_plan()

# --- Custom dt: 3, 23, 25, 27, 30 — matches 04_train_adv_no_dynamic_dt.ipynb ---
DT_DAYS_LIST = [3, 23, 25, 27, 30]       # dt_hours: 72, 552, 600, 648, 720
GPU_POOL = (7,)                          # all rollouts on cuda:7

RUN_ALL_DT = True
DT_DAYS = 25                        # only when RUN_ALL_DT=False
WEIGHT_DECAY = 1e-3
MODEL_RUN_DIR = None
# Match the Shalow sample-index strategy shown for dt=360:
# Shalow uses idx={28,138,139,207,244} (encoded in filenames).
PLOT_INDICES = [28, 138, 139, 207, 244]
DETACHED = True
# ====================================

configure_device_plan(DT_DAYS_LIST, GPU_POOL)

if not RUN_ALL_DT and DT_DAYS not in DT_DAYS_LIST:
    raise ValueError(f"DT_DAYS must be one of {DT_DAYS_LIST}")

WINDOW = get_window_spec(EXPORT_KEY)
print(f"EXPORT_KEY={EXPORT_KEY} ({WINDOW['label']}) | RUN_ALL_DT={RUN_ALL_DT}")
print(f"GPU_POOL={GPU_POOL} | DT_DAYS_LIST={DT_DAYS_LIST}")
print(format_dt_device_plan(DT_DAYS_LIST if RUN_ALL_DT else [DT_DAYS]))

# Preflight: trained model + 2023 test export per dt
models_dir = DT_DIR / "Models" / "adv_no"
run_tag = weight_decay_run_tag(RUN_TAG, WEIGHT_DECAY)
export_component = primary_export_component(EXPORT_KEY)
days = DT_DAYS_LIST if RUN_ALL_DT else [DT_DAYS]
print("\nPreflight (model + test export):")
for d in days:
    dt_h = dt_hours(d)
    rn = run_name(EXPORT_KEY, dt_hours=dt_h, run_tag=run_tag)
    rd = run_dir(models_dir, EXPORT_KEY, dt_hours=dt_h, run_tag=run_tag)
    test_p = test_prefix(DT_DIR, EXPORT_KEY, component=export_component, dt_hours=dt_h)
    model_ok = (rd / "best_model.pth").is_file()
    test_ok = Path(f"{test_p}_sims.npy").is_file()
    print(f"  dt={d:2d}d (dt{dt_h}h) model={'OK' if model_ok else 'MISSING'} test={'OK' if test_ok else 'MISSING'} → {rn}")

EXPORT_KEY=magicwin_reference (Magic window (reference)) | RUN_ALL_DT=True
GPU_POOL=(7,) | DT_DAYS_LIST=[3, 23, 25, 27, 30]
  dt= 3d → cuda:7 (CUDA_VISIBLE_DEVICES=7)
  dt=23d → cuda:7 (CUDA_VISIBLE_DEVICES=7)
  dt=25d → cuda:7 (CUDA_VISIBLE_DEVICES=7)
  dt=27d → cuda:7 (CUDA_VISIBLE_DEVICES=7)
  dt=30d → cuda:7 (CUDA_VISIBLE_DEVICES=7)

Preflight (model + test export):
  dt= 3d (dt72h) model=OK test=OK → adv_no_uv_13_22_magicwin_reference_dt72_vgg_warm10_adv60_wd1e-3
  dt=23d (dt552h) model=OK test=OK → adv_no_uv_13_22_magicwin_reference_dt552_vgg_warm10_adv60_wd1e-3
  dt=25d (dt600h) model=OK test=OK → adv_no_uv_13_22_magicwin_reference_dt600_vgg_warm10_adv60_wd1e-3
  dt=27d (dt648h) model=OK test=OK → adv_no_uv_13_22_magicwin_reference_dt648_vgg_warm10_adv60_wd1e-3
  dt=30d (dt720h) model=OK test=OK → adv_no_uv_13_22_magicwin_reference_dt720_vgg_warm10_adv60_wd1e-3


In [3]:
if RUN_ALL_DT:
    if len(GPU_POOL) == 1:
        import os
        import time
        import subprocess
        from dynamic_dt.run_registry import test_rollout_dir

        jobs = []
        print("Rollout plan (sequential on single GPU):")
        print(format_dt_device_plan(DT_DAYS_LIST))
        for dt_days in DT_DAYS_LIST:
            dt_h = dt_hours(dt_days)
            # If rollout already completed for this dt, skip to avoid re-running.
            rollout_dir = test_rollout_dir(DT_DIR, EXPORT_KEY, dt_hours=dt_h, run_tag=weight_decay_run_tag(RUN_TAG, WEIGHT_DECAY))
            if (rollout_dir / "rollout_results.json").is_file():
                print(f"\n--- rollout dt={dt_days}d --- already done (rollout_results.json exists) — skipping")
                jobs.append({"dt_days": dt_days, "device_id": None})
                continue

            print(f"\n--- rollout dt={dt_days}d ---")
            job = rollout_one_dt(
                dt_dir=DT_DIR,
                root=ROOT,
                export_key=EXPORT_KEY,
                dt_days=dt_days,
                log_dir=LOG_DIR,
                plot_indices=PLOT_INDICES,
                detached=DETACHED,
                weight_decay=WEIGHT_DECAY,
            )
            jobs.append(job)
            if DETACHED:
                pid = job["pid"]
                print(f"  waiting for PID {pid}...")
                # NOTE: `os.kill(pid, 0)` stays successful for zombie processes.
                # We also break when the process state becomes `Z`.
                while True:
                    try:
                        stat = subprocess.check_output(
                            ["ps", "-o", "stat=", "-p", str(pid)], text=True
                        ).strip()
                    except Exception:
                        stat = ""

                    if not stat:  # process disappeared
                        break
                    if stat.startswith("Z"):  # zombie / defunct
                        break

                    time.sleep(10)
                print(f"  ✓ finished dt={dt_days}d")
    else:
        jobs = launch_rollout_all(
            dt_dir=DT_DIR,
            root=ROOT,
            export_key=EXPORT_KEY,
            log_dir=LOG_DIR,
            dt_days_list=DT_DAYS_LIST,
            plot_indices=PLOT_INDICES,
            detached=DETACHED,
            weight_decay=WEIGHT_DECAY,
        )
    print_job_summary(jobs, label="Rollout")
else:
    from dynamic_dt.rollout_core import evaluate_rollout
    from dynamic_dt.run_registry import save_rollout_results, test_rollout_dir, update_comparison_index

    export_component = primary_export_component(EXPORT_KEY)
    channel_names = WINDOW["channel_names_rollout"]
    dt = dt_hours(DT_DAYS)
    data_prefix = test_prefix(DT_DIR, EXPORT_KEY, component=export_component, dt_hours=dt)
    models_dir = DT_DIR / "Models" / "adv_no"
    run_tag = weight_decay_run_tag(RUN_TAG, WEIGHT_DECAY)
    run_name_value = run_name(EXPORT_KEY, dt_hours=dt, run_tag=run_tag)
    run_dir_path = Path(MODEL_RUN_DIR) if MODEL_RUN_DIR else run_dir(
        models_dir, EXPORT_KEY, dt_hours=dt, run_tag=run_tag
    )
    test_rollout_path = test_rollout_dir(DT_DIR, EXPORT_KEY, dt_hours=dt, run_tag=run_tag)
    train_cfg, model_path, stats_path, _, _, _ = resolve_model_checkpoint(
        run_dir_path, expected_run_name=run_name_value
    )
    h1, h2, h3 = forecast_horizons_days(DT_DAYS)
    print(f"DT_DAYS={DT_DAYS} horizons +{h1}/+{h2}/+{h3}d | model={model_path}")

    if DETACHED:
        launched = rollout_one_dt(
            dt_dir=DT_DIR, root=ROOT, export_key=EXPORT_KEY, dt_days=DT_DAYS,
            log_dir=LOG_DIR, plot_indices=PLOT_INDICES, detached=True,
            weight_decay=WEIGHT_DECAY, model_run_dir=MODEL_RUN_DIR,
        )
        print(f"Started PID {launched['pid']} | tail -f {launched['log_path']}")
    else:
        from dynamic_dt.device_plan import cuda_visible_devices
        import os
        os.environ["CUDA_VISIBLE_DEVICES"] = cuda_visible_devices(DT_DAYS)
        results = evaluate_rollout(
            data_prefix=data_prefix,
            model_path=model_path,
            stats_path=stats_path,
            dt_days=DT_DAYS,
            train_cfg=train_cfg,
            output_dir=test_rollout_path,
            device="cuda:0",
            plot_indices=PLOT_INDICES,
            channel_names=channel_names,
        )
        save_rollout_results(test_rollout_path, results)
        update_comparison_index(
            models_dir, run_name_value=run_name_value, export_key=EXPORT_KEY,
            run_dir_path=run_dir_path, rollout_results=results,
        )
        print(f"Saved {test_rollout_path / 'rollout_results.json'}")

Rollout plan (sequential on single GPU):
  dt= 3d → cuda:7 (CUDA_VISIBLE_DEVICES=7)
  dt=23d → cuda:7 (CUDA_VISIBLE_DEVICES=7)
  dt=25d → cuda:7 (CUDA_VISIBLE_DEVICES=7)
  dt=27d → cuda:7 (CUDA_VISIBLE_DEVICES=7)
  dt=30d → cuda:7 (CUDA_VISIBLE_DEVICES=7)

--- rollout dt=3d --- already done (rollout_results.json exists) — skipping

--- rollout dt=23d ---
  waiting for PID 79439...


  ✓ finished dt=23d

--- rollout dt=25d ---
  waiting for PID 80287...
  ✓ finished dt=25d

--- rollout dt=27d ---
  waiting for PID 81742...
  ✓ finished dt=27d

--- rollout dt=30d ---
  waiting for PID 82493...
  ✓ finished dt=30d

Rollout — 5 job(s):
  dt=3d
  dt=23d gpu=7 PID 79439 | tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/dynamic_dt/logs/rollout_magicwin_reference_dt23d_gpu7.log
  dt=25d gpu=7 PID 80287 | tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/dynamic_dt/logs/rollout_magicwin_reference_dt25d_gpu7.log
  dt=27d gpu=7 PID 81742 | tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/dynamic_dt/logs/rollout_magicwin_reference_dt27d_gpu7.log
  dt=30d gpu=7 PID 82493 | tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/dynamic_dt/logs/rollout_magicwin_reference_dt30d_gpu7.log
